# Assignment 6.1 — NER on Pre-Tolkien Fantasy

**Thesis:** News-trained NER models systematically mis-tag fantasy named entities because their training data has no Faërie, no Oz, no Lirazel. I compare a CoNLL-2003 baseline against a Wikipedia-trained model against SpaCy's OntoNotes tagger.

**Models used:**
1. `dslim/bert-large-NER` — CoNLL-2003 baseline (PER / LOC / ORG / MISC), news-trained
2. `Babelscape/wikineural-multilingual-ner` — same 4 labels but trained on Wikipedia (more proper-noun coverage)
3. `spaCy en_core_web_sm` — OntoNotes label set (18 categories incl. WORK_OF_ART, FAC, NORP)

Plus a parallel run in Phoenix at the bottom.

**First run downloads ~2 GB; subsequent runs use the cache.**


In [1]:
from transformers import pipeline
import pandas as pd
from rich import print as rprint
from rich.text import Text

# 5 passages from the pre-Tolkien fantasy corpus, dense with invented proper nouns
passages = {
    "Dunsany — The King of Elfland's Daughter":
        "The King of Elfland's daughter Lirazel, who had crossed the frontier into Erl, longed for the fields of Faerie.",
    "Baum — The Wonderful Wizard of Oz":
        "Dorothy walked solemnly through Munchkin Country with Toto, past the Emerald City toward the castle of the Wicked Witch of the West.",
    "MacDonald — Phantastes":
        "Anodos entered the Fairy Palace, where he met the Maid of the Alder and beheld Cosmo of Prague through an enchanted mirror.",
    "Morris — The Well at the World's End":
        "Ralph of Upmeads rode through the wood of Wulstead toward the Burg of the Four Friths, seeking the Well at the World's End.",
    "Carroll — Alice's Adventures in Wonderland":
        "Alice followed the White Rabbit past the Cheshire Cat and the Mad Hatter, all the way to the court of the Queen of Hearts.",
}

# Color map for entity types (works across CoNLL and OntoNotes labels)
COLORS = {
    "PER": "red bold", "PERSON": "red bold",
    "LOC": "green bold", "GPE": "green bold", "FAC": "green bold",
    "ORG": "blue bold",
    "MISC": "yellow bold",
    "WORK_OF_ART": "magenta bold",
    "NORP": "cyan bold",
    "EVENT": "yellow bold",
}

def colorize(text, results, label_key="entity_group"):
    out = Text()
    cursor = 0
    spans = sorted(results, key=lambda r: r["start"])
    for span in spans:
        if span["start"] > cursor:
            out.append(text[cursor:span["start"]])
        kind = span[label_key].split("-")[-1].upper()
        out.append(text[span["start"]:span["end"]], style=COLORS.get(kind, "white bold"))
        cursor = span["end"]
    if cursor < len(text):
        out.append(text[cursor:])
    return out

def legend():
    t = Text()
    t.append("PER/PERSON ", style="red bold")
    t.append("LOC/GPE/FAC ", style="green bold")
    t.append("ORG ", style="blue bold")
    t.append("MISC ", style="yellow bold")
    t.append("WORK_OF_ART ", style="magenta bold")
    t.append("NORP", style="cyan bold")
    return t


/Users/caputomachine/virtual_envs/venv313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Model 1 — `dslim/bert-large-NER`

CoNLL-2003 baseline. Trained on Reuters news.  Expect it to **fail** on most invented names — that's the point of the comparison.


In [2]:
ner1 = pipeline("ner", model="dslim/bert-large-NER", aggregation_strategy="simple")

rprint(legend())
for title, text in passages.items():
    rprint(f"\n[bold magenta]{title}[/bold magenta]")
    res = ner1(text)
    rprint(colorize(text, res))
    if res:
        df = pd.DataFrame(res)[["entity_group","score","word"]]
        df["score"] = df["score"].round(3)
        print(df.to_string(index=False))
    else:
        print("(no entities found)")


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 10143.76it/s]
BertForTokenClassification LOAD REPORT from: dslim/bert-large-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PER/PERSON LOC/GPE/FAC ORG MISC WORK_OF_ART NORP

Dunsany — The King of Elfland's Daughter

The King of Elfland's daughter Lirazel, who had crossed the frontier into Erl, longed for the fields of Faerie.

entity_group  score    word
         LOC  0.969 Elfland
         PER  0.652 Lirazel
         LOC  0.969     Erl
         LOC  0.922  Faerie


...

Ralph of Upmeads rode through the wood of Wulstead toward the Burg of the Four Friths, seeking the Well at the 
World's End.

entity_group  score                    word
         PER  0.713                Ralph of
         LOC  0.570                      Up
         ORG  0.380                   ##ads
         LOC  0.941                Wulstead
         LOC  0.970 Burg of the Four Friths
         LOC  0.769                    Well
         LOC  0.981           World ' s End


Carroll — Alice's Adventures in Wonderland

Alice followed the White Rabbit past the Cheshire Cat and the Mad Hatter, all the way to the court of the Queen of 
Hearts.

entity_group  score         word
         PER  0.992        Alice
         PER  0.640 White Rabbit
         PER  0.709 Cheshire Cat
         PER  0.529   Mad Hatter
        MISC  0.340        Queen
         ORG  0.653    of Hearts


## Model 2 — `Babelscape/wikineural-multilingual-ner`

Same 4 labels (PER / LOC / ORG / MISC) but trained on **Wikipedia** rather than newswire. Wikipedia contains far more fictional proper nouns, so this should pick up names the baseline misses.


In [3]:
ner2 = pipeline("ner", model="Babelscape/wikineural-multilingual-ner", aggregation_strategy="simple")

rprint(legend())
for title, text in passages.items():
    rprint(f"\n[bold magenta]{title}[/bold magenta]")
    res = ner2(text)
    rprint(colorize(text, res))
    if res:
        df = pd.DataFrame(res)[["entity_group","score","word"]]
        df["score"] = df["score"].round(3)
        print(df.to_string(index=False))
    else:
        print("(no entities found)")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14767.89it/s]
BertForTokenClassification LOAD REPORT from: Babelscape/wikineural-multilingual-ner
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PER/PERSON LOC/GPE/FAC ORG MISC WORK_OF_ART NORP

Dunsany — The King of Elfland's Daughter

The King of Elfland's daughter Lirazel, who had crossed the frontier into Erl, longed for the fields of Faerie.

entity_group  score    word
         LOC  0.477 Elfland
         PER  0.964 Lirazel
         LOC  0.893     Erl
         LOC  0.763  Faerie


...

Ralph of Upmeads rode through the wood of Wulstead toward the Burg of the Four Friths, seeking the Well at the 
World's End.

entity_group  score                    word
         PER  0.810        Ralph of Upmeads
         LOC  0.968                Wulstead
         LOC  0.977 Burg of the Four Friths
         LOC  0.745                 Well at
        MISC  0.498                   World
         LOC  0.802                 ' s End


Carroll — Alice's Adventures in Wonderland

Alice followed the White Rabbit past the Cheshire Cat and the Mad Hatter, all the way to the court of the Queen of 
Hearts.

entity_group  score            word
         PER  0.510           Alice
        MISC  0.750    White Rabbit
        MISC  0.829    Cheshire Cat
        MISC  0.788      Mad Hatter
        MISC  0.516 Queen of Hearts


## SpaCy comparison — `en_core_web_sm` (OntoNotes labels)

SpaCy uses the **OntoNotes** label set: 18 categories including WORK_OF_ART, FAC, NORP, EVENT, LANGUAGE. Run after the HF models to compare label granularity.


In [4]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")

for title, text in passages.items():
    print(f"\n=== {title} ===")
    print(text)
    doc = nlp(text)
    print([(ent.text, ent.label_) for ent in doc.ents])



=== Dunsany — The King of Elfland's Daughter ===
The King of Elfland's daughter Lirazel, who had crossed the frontier into Erl, longed for the fields of Faerie.
[("The King of Elfland's", 'ORG'), ('Lirazel', 'PERSON'), ('Erl', 'PERSON'), ('Faerie', 'GPE')]

=== Baum — The Wonderful Wizard of Oz ===
Dorothy walked solemnly through Munchkin Country with Toto, past the Emerald City toward the castle of the Wicked Witch of the West.
[('Dorothy', 'PERSON'), ('Munchkin Country', 'ORG'), ('Emerald City', 'GPE'), ('the Wicked Witch of the West', 'ORG')]

=== MacDonald — Phantastes ===
Anodos entered the Fairy Palace, where he met the Maid of the Alder and beheld Cosmo of Prague through an enchanted mirror.
[('the Fairy Palace', 'EVENT'), ('Alder', 'PERSON'), ('Cosmo', 'ORG'), ('Prague', 'GPE')]

=== Morris — The Well at the World's End ===
Ralph of Upmeads rode through the wood of Wulstead toward the Burg of the Four Friths, seeking the Well at the World's End.
[('Ralph of Upmeads', 'PERSON')

In [5]:
# Inline color-coded render — screenshot this for the assignment
docs = [nlp(text) for text in passages.values()]
displacy.render(docs, style="ent", jupyter=True)


## Phoenix / LLM comparison

**Run this prompt in Phoenix (or any LLM), screenshot the reply, and paste the entities back below:**

> For each numbered passage below, list every named entity and label it (PERSON, LOCATION, ORGANIZATION, WORK_OF_ART, EVENT, etc.). Return one bullet list per passage.
>
> 1. The King of Elfland's daughter Lirazel, who had crossed the frontier into Erl, longed for the fields of Faerie.
> 2. Dorothy walked solemnly through Munchkin Country with Toto, past the Emerald City toward the castle of the Wicked Witch of the West.
> 3. Anodos entered the Fairy Palace, where he met the Maid of the Alder and beheld Cosmo of Prague through an enchanted mirror.
> 4. Ralph of Upmeads rode through the wood of Wulstead toward the Burg of the Four Friths, seeking the Well at the World's End.
> 5. Alice followed the White Rabbit past the Cheshire Cat and the Mad Hatter, all the way to the court of the Queen of Hearts.

### Paragraph (write this after seeing all four outputs)

Template to fill in:

> The CoNLL-trained baseline `dslim/bert-large-NER` systematically dropped or mis-labeled invented names: it labeled \_\_ as \_\_ and missed \_\_ entirely. `Babelscape/wikineural` recovered \_\_ thanks to its Wikipedia training data, but still struggled with \_\_. SpaCy's OntoNotes label set was the most informative for fantasy because labels like WORK_OF_ART and FAC actually fit "the Emerald City" and "the Well at the World's End." Phoenix outperformed all three because \_\_ — though it did not return calibrated confidence scores. My own reading would have tagged \_\_ which none of the models caught, because \_\_.
